# 05 — PACE defence: planner + CFI gate + K-of-q quorum

PACE = **Plan, Agree, Confirm, Execute**. Walk-through of the headline defence.

1. The Planner emits a finite list of allowed `(tool, arg-schema)` entries (the *shadow plan* π̂) before any untrusted byte enters the loop.
2. Every proposed tool call is checked against π̂ by the **CFI gate** (set-membership on tool name + argument shape).
3. K Executors run in parallel over disjoint NLI-clustered evidence; the **quorum gate** fires the call only when at least q of K agree on the same canonical (tool, args).
4. An optional **NLI redundancy filter** drops entailment-redundant evidence spans before clustering so a single semantic message cannot occupy multiple clusters.

This notebook exercises the pure-Python core (CFI gate + AgreementVoter) without any model calls so it runs anywhere.

In [1]:
from aaps.defenses.pace.plan import (
    PACEPlan, PACEPlanNode, ProposedToolCall, canonicalise_args
)
from aaps.defenses.pace.agreement import AgreementVoter

print('PACE building blocks imported.')

/Users/tlysu/ucu/Diploma/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PACE building blocks imported.


## 1. Build a shadow plan π̂

In [2]:
plan = PACEPlan(
    nodes=[
        PACEPlanNode(tool='send_email', arg_schema={'to': 'str', 'body': 'str'}),
        PACEPlanNode(tool='search_docs', arg_schema={'query': 'str'}),
    ],
    user_request='please email alice'
)
print('plan_id:', plan.plan_id)
print('nodes:', [n.tool for n in plan.nodes])

plan_id: 75c94f59f318
nodes: ['send_email', 'search_docs']


## 2. CFI gate: in-plan call passes, out-of-plan blocked

In [3]:
node = plan.find_node('send_email', {'to': 'alice@example.com', 'body': 'hi'})
print('send_email in plan:', node is not None)

blocked = plan.find_node('delete_file', {'path': '/etc/passwd'})
print('delete_file in plan:', blocked is not None)

send_email in plan: True
delete_file in plan: False


## 3. Agreement voter: quorum on canonical (tool, args)

K=5 Executors over 5 evidence clusters; q=3 required to fire.

In [4]:
voter = AgreementVoter(K=5, q=3)
alice = [ProposedToolCall(tool='send_email', args={'to': 'alice@example.com'}, cluster_id=i) for i in range(3)]
attacker = [ProposedToolCall(tool='send_email', args={'to': 'attacker@evil.io'}, cluster_id=3+i) for i in range(2)]
per_cluster = [[c] for c in alice] + [[c] for c in attacker]
decisions = voter.vote(per_cluster)
for d in decisions:
    fires = d.agreement >= d.q
    print(f'tool={d.tool} to={d.args.get("to")!s:35} support={d.agreement}/{d.K}  -> {"FIRE" if fires else "BLOCK"}')

tool=send_email to=alice@example.com                   support=3/5  -> FIRE
tool=send_email to=attacker@evil.io                    support=2/5  -> BLOCK


## 4. canonicalise_args is order-insensitive

In [5]:
a = canonicalise_args('send_email', {'to': 'a@b.c', 'body': 'hi'})
b = canonicalise_args('send_email', {'body': 'hi', 'to': 'a@b.c'})
print('same key for permuted args:', a == b)

same key for permuted args: True


## 5. Full pipeline (with an LLM)

For an end-to-end run with a real Planner + Executor see `06_benchmark_comparison.ipynb` and `08_agentdojo_benchmark.ipynb`.